# Sensor.Community Data Ingestion

Polls the Sensor.Community API for the latest 5-minute air quality readings
and writes them as JSON files to a Unity Catalog Volume for Auto Loader pickup.

This notebook runs on a schedule (every 30 min) as a Lakeflow Job.
It is NOT part of the SDP pipeline.

In [ ]:
import requests
import json
from datetime import datetime

# Sensor.Community API - last 5 min of dust sensor readings (PM10/PM2.5)
# Filtered to a specific area to keep data volume manageable on Free Edition
# This fetches sensors within 50km of Berlin (major Sensor.Community hub)
API_URL = "https://data.sensor.community/airrohr/v1/filter/area=52.52,13.405,50"

VOLUME_PATH = "/Volumes/catalog1/iot_hello_world/raw_landing/sensor_data"

headers = {"User-Agent": "databricks-iot-hello-world/1.0"}

In [ ]:
# Fetch latest sensor readings
response = requests.get(API_URL, headers=headers, timeout=60)
response.raise_for_status()
raw_data = response.json()
print(f"Fetched {len(raw_data)} sensor readings")

In [ ]:
# Flatten nested sensor data into one record per reading
fetch_ts = datetime.utcnow().isoformat()
records = []

for entry in raw_data:
    sensor = entry.get("sensor", {})
    location = entry.get("location", {})

    # Extract sensor data values into a flat dict
    values = {}
    for sdv in entry.get("sensordatavalues", []):
        values[sdv["value_type"]] = sdv["value"]

    records.append({
        "sensor_id": sensor.get("id"),
        "sensor_type": sensor.get("sensor_type", {}).get("name"),
        "location_id": location.get("id"),
        "latitude": location.get("latitude"),
        "longitude": location.get("longitude"),
        "country": location.get("country"),
        "timestamp": entry.get("timestamp"),
        "P1": values.get("P1"),
        "P2": values.get("P2"),
        "temperature": values.get("temperature"),
        "humidity": values.get("humidity"),
        "pressure": values.get("pressure"),
        "fetch_ts": fetch_ts,
    })

print(f"Flattened to {len(records)} records")

In [ ]:
# Write as a single JSON file with timestamp in filename
# Auto Loader will pick up new files incrementally
file_ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
file_path = f"{VOLUME_PATH}/{file_ts}.json"

# Write as newline-delimited JSON (one record per line)
# This is the format Auto Loader expects with .json
ndjson = "
".join(json.dumps(r) for r in records)
dbutils.fs.put(file_path, ndjson, overwrite=True)

print(f"Wrote {len(records)} records to {file_path}")